# Phase 7: Explainable AI and Legal Retrieval Interpretability

**Project**: Enhanced Legal Document Similarity and Case Retrieval Using Hybrid BERT, Legal Knowledge, and Ensemble Learning  
**Target Architecture**: Validated Weighted Linear Score Fusion ($w_{\text{bm25}} = 0.00, w_{\text{tfidf}} = 0.65, w_{\text{dense}} = 0.25, w_{\text{legal}} = 0.10$)  
**Objective**: Implement an auditable, domain-specific explainability framework explaining why candidates are retrieved, decomposing exact score contributions, generating closed-form SHAP attributions, producing factual case explanations, and tracking multi-model ranking transitions.

In [ ]:
import os
import sys
import json
import tempfile
from pathlib import Path

os.environ["MPLCONFIGDIR"] = tempfile.gettempdir()
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.explainability.feature_attribution import FeatureAttributor
from src.explainability.shap_explainer import LinearSHAPExplainer
from src.explainability.case_explanation import CaseExplanationGenerator
from src.explainability.ranking_comparison import RankingComparisonAnalyzer

print('All explainability modules loaded successfully.')

## 1. Load Frozen Checkpoint and Development Features

We load the frozen development features (`hybrid_features.parquet`) with 120,596 pairs across 118 queries and verify the frozen checkpoint weights.

In [ ]:
ckpt_path = PROJECT_ROOT / 'models/checkpoints/weighted_fusion_weights.json'
with open(ckpt_path, 'r') as f:
    ckpt = json.load(f)
print(f"Frozen Model Configuration: {ckpt}")

dev_path = PROJECT_ROOT / 'data/processed/IL-TUR-PCR/dev/hybrid_features.parquet'
df_dev = pd.read_parquet(dev_path)
print(f"Loaded Development Split: {len(df_dev):,} candidate pairs, {df_dev['query_id'].nunique()} queries, {df_dev['label'].sum()} positive citations.")

## 2. Exact Mathematical Score Decomposition

In Weighted Linear Score Fusion, the hybrid score is an exact additive linear combination:
$$\hat{S}_{\text{hybrid}}(d) = 0.65 \cdot \tilde{S}_{\text{tfidf}}(d) + 0.25 \cdot \tilde{S}_{\text{inlegalbert}}(d) + 0.10 \cdot \tilde{S}_{\text{legal}}(d)$$
The sum of modal contributions identically equals the predicted hybrid score with zero residual.

In [ ]:
attributor = FeatureAttributor(
    feature_weights={'tfidf_score': 0.65, 'inlegalbert_sim': 0.25, 'composite_legal_score': 0.10, 'bm25_score': 0.00},
    use_query_min_max=True
)
df_attr = attributor.compute_attributions(df_dev)

# Verify zero numerical residual
reconstructed = df_attr['contrib_tfidf_score'] + df_attr['contrib_inlegalbert_sim'] + df_attr['contrib_composite_legal_score']
max_residual = (df_attr['predicted_hybrid_score'] - reconstructed).abs().max()
print(f"Maximum Additive Decomposition Residual: {max_residual:.10f}")

# Slice summaries
df_modality = attributor.summarize_modality_contributions(df_attr)
display(df_modality)

## 3. Closed-Form Linear SHAP Explainability

Because the retrieval model is a linear score fusion rather than a tree ensemble, TreeSHAP was rejected. We compute closed-form linear Shapley values:
$$\phi_i(x) = w_i \cdot (x_i - \mathbb{E}[X_i])$$
satisfying the efficiency axiom $\sum_i \phi_i = f(x) - \mathbb{E}[f(X)]$.

In [ ]:
shap_explainer = LinearSHAPExplainer(
    weights={'tfidf_score': 0.65, 'inlegalbert_sim': 0.25, 'composite_legal_score': 0.10, 'bm25_score': 0.00},
    feature_names=['tfidf_score', 'inlegalbert_sim', 'composite_legal_score', 'bm25_score'],
    background_data=df_dev.sample(500, random_state=42)
)

shap_vals, base_val = shap_explainer.explain_dataset(df_dev)
shap_importances = shap_explainer.get_global_feature_importance(shap_vals)
print(f"Base Value (Expected Hybrid Score): {base_val:.5f}")
display(shap_importances)

## 4. Case-Level Factual Explanations

We generate structured, verifiable explanations for retrieved candidate judgments, inspecting shared statutory provisions (IPC, CrPC), citations, judicial coram, and rhetorical similarity.

In [ ]:
query_legal_path = PROJECT_ROOT / 'data/processed/IL-TUR-PCR/dev_queries/legal_features.json'
cand_legal_path = PROJECT_ROOT / 'data/processed/IL-TUR-PCR/dev_candidates/legal_features.json'

case_gen = CaseExplanationGenerator(str(query_legal_path), str(cand_legal_path))

# Sample top retrieved candidate for query 0000012637
sample_qid = '0000012637'
top_pair = df_attr[df_attr['query_id'] == sample_qid].sort_values('predicted_hybrid_score', ascending=False).iloc[0]

explanation = case_gen.generate_explanation(sample_qid, top_pair['candidate_id'], top_pair.to_dict())
print(json.dumps(explanation, indent=2))

## 5. Ranking Transition & Movement Dynamics

We analyze how hybrid fusion reconciles lexical, dense, and legal signals to promote true precedents and demote lexical false alarms.

In [ ]:
rank_analyzer = RankingComparisonAnalyzer()
comparison = rank_analyzer.compare_query_rankings(df_attr, sample_qid)

print(f"Query {sample_qid}: Identified {len(comparison['promoted_precedents'])} promoted precedents and {len(comparison['demoted_false_positives'])} demoted false alarms.")
if comparison['promoted_precedents']:
    print("\n--- Promoted Precedent ---")
    print(json.dumps(comparison['promoted_precedents'][0], indent=2))
if comparison['demoted_false_positives']:
    print("\n--- Demoted False Positive ---")
    print(json.dumps(comparison['demoted_false_positives'][0], indent=2))

## 6. Visualizations

Displaying the generated high-resolution (300 DPI) publication figures.

In [ ]:
from IPython.display import Image, display

fig_paths = [
    PROJECT_ROOT / 'results/figures/global_feature_importance.png',
    PROJECT_ROOT / 'results/figures/modality_contributions.png',
    PROJECT_ROOT / 'results/figures/shap_summary_plot.png',
    PROJECT_ROOT / 'results/figures/score_decomposition_examples.png',
    PROJECT_ROOT / 'results/figures/ranking_movement_slope.png'
]

for p in fig_paths:
    if p.exists():
        print(f"Displaying: {p.name}")
        display(Image(filename=str(p)))
    else:
        print(f"Missing: {p.name}")